# Sentiment Analysis 

### Naive Bayes 
- Gaussian Naive Bayes (GaussianNB)
- Multinomial Naive Bayes (MultinomialNB)
- Bernoulli Naive Bayes (BernoulliNB)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.describe

<bound method NDFrame.describe of                                                   review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]>

In [5]:
df ["review"][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

## Text data cleaning 
1. Sample 10000 rows 
2. Remove html tags 
3. Remove special characters 
4. Converting every thing to lower case 
5. Removing stop words 
6. Stamming


In [6]:
df= df.sample(10000)

In [7]:
df.info

<bound method DataFrame.info of                                                   review sentiment
25198  This star-studded British/Spanish co-productio...  negative
41705  ....shut it off. The prologue with Fu Manchu's...  negative
17844  **POSSIBLE SPOILERS**<br /><br />The biggest p...  positive
15144  Think of this pilot as "Hawaii Five-O Lite". I...  negative
37823  In a future where an industrious travel agency...  negative
...                                                  ...       ...
41335  This film was made and cast from my home town....  negative
21845  I'm surprised over the number of folks that ha...  negative
21288  I know this film was shown on local TV when I ...  negative
19783  Not for the squeamish, but the number of twist...  positive
47858  I remember seeing promos for this show before ...  positive

[10000 rows x 2 columns]>

In [8]:
# df['sentiment'].replace({"positive": 1 , "negative":0},inplace= True)  # it may automatic downcasting

df['sentiment'] = df['sentiment'].replace({
    "positive": 1,
    "negative": 0
})
# the second version is the recommended modern Pandas style.

/tmp/ipykernel_16/928976059.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({


In [9]:
df.head()

,review,sentiment
25198,This star-studded British/Spanish co-productio...,0
41705,....shut it off. The prologue with Fu Manchu's...,0
17844,**POSSIBLE SPOILERS**<br /><br />The biggest p...,1
15144,"Think of this pilot as ""Hawaii Five-O Lite"". I...",0
37823,In a future where an industrious travel agency...,0


In [10]:
"""
Removing HTML tags from the review text using Regular Expressions (regex). The regex pattern '<.*?>' matches any substring that starts with '<', followed by any characters (non-greedy), and ends with '>'. This effectively captures HTML tags, which can then be removed from the text.

"""

import re # Regular Expression

clean  = re.compile('<.*?>')  # regex to remove html tags
re.sub(clean, '', df.iloc[2].review ) # "Take the review in the 3rd row, find all HTML tags, and replace them with nothing."

"**POSSIBLE SPOILERS**The biggest part of the movie that doesn't work IS the Wendigo, and when your title character fails, your movie usually isn't far behind it. The filmmakers' interpretation of the Wendigo's form is interesting, and can be properly menacing when filmed correctly - when the fleeing killer sees the Wendigo in a flash in his rear view mirror, for instance - and the tree-form was actually very good. However, as a monster character it never really comes to life. We don't get much of an explanation for its behavior, and what we DO see from it doesn't jibe with either the story told in the movie itself, or any Wendigo lore I've ever read.I think one of the main reasons that the monster fails is that it isn't given enough to do, in the movie. When you boil this film down to its bones, what you have is a suspense thriller with a little bit of a supernatural element, instead of a movie about a monster.The cinematography is good, though a little cheesy; the filmmakers use scen

In [11]:
# function to clean the review text by removing HTML tags 
def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)  

In [12]:
df['review'] = df['review'].apply(clean_html)

In [13]:
# conver lower case
def convert_lower(text):
    return text.lower()

In [14]:
df['review'] = df['review'].apply(convert_lower)

In [15]:
# function to remove special characters from the review text using Regular Expressions (regex). The regex pattern '[^a-zA-Z0-9\s]' matches any character that is not a letter (uppercase or lowercase), a digit, or whitespace. This effectively captures special characters, which can then be removed from the text.
def remove_special_characters(text):
    x=''
    for i in text:
        if i.isalnum() or i.isspace():
            x+=i
        else:
            x+=' '

    return x

In [16]:
df['review'] = df['review'].apply(remove_special_characters)

In [17]:
# remove stop words from the review text using the Natural Language Toolkit (nltk). Stop words are common words that are often removed from text data to improve the performance of natural language processing models. The function takes a string of text as input, splits it into individual words, filters out any stop words, and then joins the remaining words back into a single string.
import nltk
from nltk.corpus import stopwords
def remove_stop_words(text):
    x=[]
    for i in text.split():
        if i in stopwords.words('english'):
            x.append(i)

    y=x[:]
    x.clear()
    return y

In [18]:
df['review'] = df['review'].apply(remove_special_characters)

In [19]:
df["review"] = df["review"].str.split()
df["review"].iloc[0]


['this',
 'star',
 'studded',
 'british',
 'spanish',
 'co',
 'production',
 'looks',
 'great',
 'what',
 'you',
 'can',
 'see',
 'of',
 'it',
 'i',
 'have',
 'three',
 'versions',
 'two',
 'vhs',
 'one',
 'dvd',
 'and',
 'all',
 'are',
 'terribly',
 'cropped',
 'so',
 'badly',
 'that',
 'it',
 'looks',
 'as',
 'if',
 'buildings',
 'are',
 'having',
 'conversations',
 'with',
 'each',
 'other',
 'few',
 'films',
 'suffer',
 'as',
 'badly',
 'from',
 'pan',
 'and',
 'scan',
 'as',
 'this',
 'one',
 'as',
 'director',
 'robert',
 'parrish',
 'seems',
 'to',
 'have',
 'been',
 'so',
 'enamored',
 'with',
 'the',
 'widescreen',
 'process',
 'that',
 'he',
 'tended',
 'to',
 'use',
 'both',
 'sides',
 'of',
 'the',
 'screen',
 'at',
 'once',
 'neglecting',
 'the',
 'middle',
 'another',
 'user',
 'comments',
 'that',
 'we',
 'see',
 'the',
 'entire',
 'inhabitants',
 'of',
 'a',
 'church',
 'massacred',
 'at',
 'the',
 'beginning',
 'not',
 'in',
 'any',
 'of',
 'the',
 'copies',
 'i',
 'ha

In [20]:
df.head()

,review,sentiment
25198,"[this, star, studded, british, spanish, co, pr...",0
41705,"[shut, it, off, the, prologue, with, fu, manch...",0
17844,"[possible, spoilers, the, biggest, part, of, t...",1
15144,"[think, of, this, pilot, as, hawaii, five, o, ...",0
37823,"[in, a, future, where, an, industrious, travel...",0


In [21]:
# Performing sentiment analysis on the cleaned review text using the TextBlob library. The function takes a string of text as input, creates a TextBlob object, and then calculates the sentiment polarity of the text. The polarity score ranges from -1 (negative sentiment) to 1 (positive sentiment), with 0 indicating neutral sentiment. The function returns the polarity score as a float.
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [22]:
y=[]
def stem_words(text):
    for i in text:
        y.append(ps.stem(i))
    z= y[:]
    y.clear()
    return z

In [23]:
stem_words(["I","loved", "loving","it"])

['i', 'love', 'love', 'it']

In [24]:
df['review'] = df['review'].apply(stem_words)

In [25]:
# join back
def join_back(text):
    return ' '.join(text)

In [26]:
df['review'] = df['review'].apply(join_back)

In [27]:
df.review.head()

25198    thi star stud british spanish co product look ...
41705    shut it off the prologu with fu manchu s birth...
17844    possibl spoiler the biggest part of the movi t...
15144    think of thi pilot as hawaii five o lite it s ...
37823    in a futur where an industri travel agenc use ...
Name: review, dtype: object

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [29]:
X=cv.fit_transform(df['review']).toarray()

In [30]:
X.shape

(10000, 36302)

In [31]:
y = df.iloc[:, -1].values 

In [32]:
y.shape

(10000,)

In [33]:
y

array([0, 0, 1, ..., 0, 1, 1])

In [34]:
# Split the dataset into training and testing sets using sklearn's train_test_split function.
# The test size is set to 20% of the dataset, and a random state is provided for reproducibility.
from sklearn.model_selection import train_test_split

In [35]:
X_train , X_test , y_train , y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [36]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 36302), (2000, 36302), (8000,), (2000,))

In [37]:
from sklearn.naive_bayes import GaussianNB ,MultinomialNB, BernoulliNB

In [38]:
clf1 = GaussianNB()
clf2 = MultinomialNB()
clf3 = BernoulliNB()

In [39]:
clf1.fit(X_train , y_train)

GaussianNB()

In [40]:
clf2.fit(X_train , y_train)

MultinomialNB()

In [41]:
clf3.fit(X_train , y_train)

BernoulliNB()